In [1]:
import os
import sys
import json
import pandas as pd

sys.path.append(os.path.join(os.getcwd(), ".."))

from src.preprocessing import clean_text, drop_empty, split_and_save

BASE_DIR = os.path.dirname(os.getcwd())
RAW_DIR  = os.path.join(BASE_DIR, "data", "raw")
PROC_DIR = os.path.join(BASE_DIR, "data", "processed")
os.makedirs(PROC_DIR, exist_ok=True)

#### DATASET 1 – NEWS HEADLINES

In [2]:
headlines_path = os.path.join(RAW_DIR, "newsheadline", "Sarcasm_Headlines_Dataset_v2.json")

records = []
with open(headlines_path, "r") as f:
    for line in f:
        records.append(json.loads(line))

headlines_df = pd.DataFrame(records)
print("Headlines raw shape:", headlines_df.shape)
print(headlines_df.head(3))

Headlines raw shape: (28619, 3)
   is_sarcastic                                           headline  \
0             1  thirtysomething scientists unveil doomsday clo...   
1             0  dem rep. totally nails why congress is falling...   
2             0  eat your veggies: 9 deliciously different recipes   

                                        article_link  
0  https://www.theonion.com/thirtysomething-scien...  
1  https://www.huffingtonpost.com/entry/donna-edw...  
2  https://www.huffingtonpost.com/entry/eat-your-...  


In [3]:
headlines_df = headlines_df[["headline", "is_sarcastic"]].rename(
    columns={"headline": "text", "is_sarcastic": "label"}
)
headlines_df["text"]  = headlines_df["text"].apply(clean_text)
headlines_df["label"] = headlines_df["label"].astype(int)
headlines_df = drop_empty(headlines_df, "text")

print("\nClass distribution:")
print(headlines_df["label"].value_counts())
print("\nSample:")
print(headlines_df.head(3).to_string())


Class distribution:
0    14985
1    13634
Name: label, dtype: int64

Sample:
                                                                              text  label
0                    thirtysomething scientists unveil doomsday clock of hair loss      1
1  dem rep. totally nails why congress is falling short on gender, racial equality      0
2                                eat your veggies: 9 deliciously different recipes      0


In [4]:
hl_train, hl_test = split_and_save(headlines_df, PROC_DIR, "headlines")

headlines → train: 22895, test: 5724


#### DATASET 2 – REDDIT SARC

In [5]:
reddit_path = os.path.join(RAW_DIR, "sarc_reddit", "train-balanced-sarcasm.csv")

reddit_df = pd.read_csv(reddit_path)
print("Reddit raw shape:", reddit_df.shape)
print("Columns:", reddit_df.columns.tolist())
print(reddit_df.head(3).to_string())

Reddit raw shape: (1010826, 10)
Columns: ['label', 'comment', 'author', 'subreddit', 'score', 'ups', 'downs', 'date', 'created_utc', 'parent_comment']
   label                                                                                                                    comment     author subreddit  score  ups  downs     date          created_utc                                                                                                                          parent_comment
0      0                                                                                                                 NC and NH.  Trumpbart  politics      2   -1     -1  2016-10  2016-10-16 23:55:23                                                        Yeah, I get that argument. At this point, I'd prefer is she lived in NC as well.
1      0                                                 You do know west teams play against west teams more than east teams right?  Shbshb906       nba     -4   -1     -1  

In [6]:
reddit_df = reddit_df[["label", "comment", "parent_comment", "subreddit"]].copy()
reddit_df.rename(columns={"comment": "text", "parent_comment": "context"}, inplace=True)

reddit_df["text"]    = reddit_df["text"].fillna("").apply(clean_text)  # fillna BEFORE clean
reddit_df["context"] = reddit_df["context"].fillna("").apply(clean_text)
reddit_df["label"]   = reddit_df["label"].astype(int)
reddit_df = drop_empty(reddit_df, "text")

In [7]:
# Context column for RQ2
reddit_df["text_with_context"] = reddit_df.apply(
    lambda row: row["context"] + " [SEP] " + row["text"]
    if row["context"] != ""
    else row["text"],
    axis=1
)

In [8]:
rd_train, rd_test = split_and_save(reddit_df, PROC_DIR, "reddit")

reddit → train: 808612, test: 202153


#### DATASET 3 – SemEval 2018

In [9]:
semeval_path = os.path.join(
    RAW_DIR, "semeval",
    "SemEval2018-T3-train-taskA_emoji_ironyHashtags.txt"
)

semeval_df = pd.read_csv(
    semeval_path,
    sep="\t",
    skiprows=1,
    header=None,
    names=["index", "label", "text"]
)
print("SemEval raw shape:", semeval_df.shape)
print(semeval_df.head(3).to_string())

SemEval raw shape: (3817, 3)
   index  label                                                                                                                                    text
0      1      1                              Sweet United Nations video. Just in time for Christmas. #imagine #NoReligion #irony http://t.co/fej2v3OUBR
1      2      1  @mrdahl87 We are rumored to have talked to Erv's agent... and the Angels asked about Ed Escobar... that's hardly nothing #Sarcasm   ;)
2      3      1                                                                             Hey there! Nice to see you Minnesota/ND Winter Weather #Not


In [10]:
semeval_df["text"]  = semeval_df["text"].apply(clean_text)
semeval_df["label"] = semeval_df["label"].astype(int)
semeval_df.drop(columns=["index"], inplace=True)
semeval_df = drop_empty(semeval_df, "text")

print("\nClass distribution:")
print(semeval_df["label"].value_counts())
print("\nSample:")
print(semeval_df.head(3).to_string())


Class distribution:
0    1915
1    1901
Name: label, dtype: int64

Sample:
   label                                                                                                                       text
0      1                                           sweet united nations video. just in time for christmas. imagine noreligion irony
1      1  we are rumored to have talked to erv's agent... and the angels asked about ed escobar... that's hardly nothing sarcasm ;)
2      1                                                                 hey there! nice to see you minnesota/nd winter weather not


In [11]:
se_train, se_test = split_and_save(semeval_df, PROC_DIR, "semeval")

semeval → train: 3052, test: 764


In [12]:
print("PREPROCESSING COMPLETE")

for name, df in [("Headlines", headlines_df), ("Reddit SARC", reddit_df), ("SemEval", semeval_df)]:
    total    = len(df)
    sarc     = df["label"].sum()
    non_sarc = total - sarc
    avg_len  = df["text"].str.split().str.len().mean()
    print(f"\n{name}")
    print(f"  Total samples : {total}")
    print(f"  Sarcastic (1) : {sarc}")
    print(f"  Non-sarc  (0) : {non_sarc}")
    print(f"  Avg text len  : {avg_len:.1f} words")

PREPROCESSING COMPLETE

Headlines
  Total samples : 28619
  Sarcastic (1) : 13634
  Non-sarc  (0) : 14985
  Avg text len  : 10.1 words

Reddit SARC
  Total samples : 1010818
  Sarcastic (1) : 505411
  Non-sarc  (0) : 505407
  Avg text len  : 10.5 words

SemEval
  Total samples : 3816
  Sarcastic (1) : 1901
  Non-sarc  (0) : 1915
  Avg text len  : 13.5 words
